In [1]:
# Feature engineering; see docs/project-update/features/04-feature-engineering-zh.md
# Run order: 01 flag_anomalies -> 02 fill_missing -> 03 handle_outliers -> make_features. -i lets 03 see the functions from 01 and 02.
%run 01_flag_anomalies.ipynb
%run 02_missing_values.ipynb
%run -i 03_outliers.ipynb

31370 rows, default rate 5.73% vs 4.33%


In [2]:
import numpy as np
import pandas as pd

# Features from the project proposal (docs/Turquoise-Project Proposal -RMS_edits-flat.pdf).
# Left out on purpose: interest coverage, financial expenses, financial income (exp_financing / inc_financing),
# the NLP signals, the raw amounts the proposal says are not used directly, and ROE (the roe column is dropped in 02).
INDUSTRY_GROUPS = ["primary_utilities", "manufacturing", "construction", "trade", "transport_hospitality", "real_estate", "services_other"]
LEGAL_STRUCTS = ["SRL", "SRU", "SPA", "SAU", "SAA", "SRS"]  # categories seen in training; an unseen one gets all zeros


def _industry_group(sector):
    # ATECO 2007 divisions grouped by section: A,B,D,E | C | F | G | H,I | L | the rest (J, M, N, O-S, unknown)
    conds = [sector.between(1, 9) | sector.between(35, 39), sector.between(10, 33), sector.between(41, 43),
             sector.between(45, 47), sector.between(49, 56), sector == 68]
    return pd.Series(np.select(conds, INDUSTRY_GROUPS[:6], "services_other"), index=sector.index)


def _ratio(num, den):
    # A denominator <= 0 gives NaN instead of inf or a meaningless negative ratio
    return num / den.where(den > 0)


def make_features(df):
    """Take data that has been through 01, 02 and 03, return only the proposal features (same index as the input)."""
    debt = df["debt_st"] + df["debt_lt"]
    f = pd.concat([
        pd.get_dummies(_industry_group(df["ateco_sector"]).astype(pd.CategoricalDtype(INDUSTRY_GROUPS)), prefix="industry", dtype=int),
        pd.get_dummies(df["legal_struct"].where(df["legal_struct"].isin(LEGAL_STRUCTS)).astype(pd.CategoricalDtype(LEGAL_STRUCTS)), prefix="legal", dtype=int),
    ], axis=1)

    f["size"] = np.log(df["asst_tot"].where(df["asst_tot"] > 0))

    # Solvency and capital structure
    f["equity_ratio"] = df["eqty_tot"] / df["asst_tot"]
    f["debt_ratio"] = debt / df["asst_tot"]
    f["neg_equity_flag"] = (df["eqty_tot"] < 0).astype(int)
    f["st_debt_share"] = _ratio(df["debt_st"], debt)
    f["bank_debt_concentration"] = _ratio(df["debt_bank_st"] + df["debt_bank_lt"], debt)

    # Liquidity and working capital
    f["cash_ratio"] = df["cash_and_equiv"] / df["asst_tot"]
    f["current_assets_ratio"] = df["asst_current"] / df["asst_tot"]
    f["receivables_ratio"] = _ratio(df["AR"], df["rev_operating"])
    f["payables_ratio"] = df["AP_st"] / df["asst_tot"]
    f["working_capital_ratio"] = df["wc_net"] / df["asst_tot"]
    f["current_ratio"] = _ratio(df["asst_current"], df["debt_st"])

    # Profitability and operating performance
    f["roa"] = df["roa"]
    f["ebitda_ratio"] = df["ebitda"] / df["asst_tot"]
    f["operating_margin"] = _ratio(df["prof_operations"], df["rev_operating"])
    f["loss_flag"] = (df["profit"] < 0).astype(int)

    # Debt-servicing and cash-flow capacity
    f["cash_flow_to_debt"] = _ratio(df["cf_operations"], debt)
    f["debt_to_ebitda"] = _ratio(debt, df["ebitda"])
    return f

In [3]:
filled = fill_missing(flag_anomalies(data), sector_medians(data))
feats = make_features(handle_outliers(filled))

# Check: same rows, no inf, one-hot columns are exclusive, NaN only where the denominator is not positive
assert feats.index.equals(data.index)
assert not np.isinf(feats).any().any()
assert (feats.filter(like="industry_").sum(axis=1) == 1).all()
assert (feats.filter(like="legal_").sum(axis=1) == 1).all()
assert feats["receivables_ratio"].isna().sum() == (filled["rev_operating"] <= 0).sum()
assert feats["current_ratio"].isna().sum() == (filled["debt_st"] <= 0).sum()
assert feats["neg_equity_flag"].sum() == (filled["eqty_tot"] < 0).sum()

# Check: a legal structure never seen in training, and a missing sector, do not break it
raw = filled.head(2).copy()
raw["legal_struct"] = ["XXX", "SRL"]
raw.loc[raw.index[0], "ateco_sector"] = np.nan
out = make_features(raw)
assert out.filter(like="legal_").iloc[0].sum() == 0 and out["industry_services_other"].iloc[0] == 1

pd.DataFrame({"nan_pct": (feats.isna().mean() * 100).round(2), "median": feats.median().round(3)})

,nan_pct,median
industry_primary_utilities,0.00,0.000
industry_manufacturing,0.00,0.000
industry_construction,0.00,0.000
industry_trade,0.00,0.000
industry_transport_hospitality,0.00,0.000
industry_real_estate,0.00,0.000
industry_services_other,0.00,0.000
legal_SRL,0.00,1.000
legal_SRU,0.00,0.000
legal_SPA,0.00,0.000
